# exercise_2.1_Loading & First Diagnostics (Datania Households)

This exercise uses `datania_households_raw.csv` and focuses on first-pass loading checks, data diagnostics, and column selection strategy.

### Path Setup (run first)

In [ ]:
import os
import pandas as pd

# Works whether kernel cwd is project root or notebooks/
DATA_RAW_DIR = '../../data/0_raw'


---

## Task 1 — Load the raw CSV file

Load the household file into a DataFrame.

In [ ]:
df = pd.read_csv(os.path.join(DATA_RAW_DIR , 'datania_households_raw.csv'))
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

**Questions:**

- Did the file load without errors?
- What does each row represent in this dataset?
- Which columns already look like potential data-quality risks just from the names?

---

## Task 2 — First visual inspection

Before running statistics, inspect raw records manually.

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.sample(10, random_state=42)

**Questions:**

- Can you spot suspicious values by eye (for example in `income_dkw`, `survey_date`, `urban_rural`, `hh_size`, `age`)?
- Do you see possible duplicates in `hh_id`?
- Write down 3 issues to investigate further.

---

## Task 3 — Run `df.info()` and missing diagnostics

Inspect data types and non-null counts for every column.

In [ ]:
df.info(verbose=True)

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

**Questions:**

- Which columns are `object` but should likely be numeric or dates?
- Which columns contain missing values, and how many?
- Pick 3 columns where dtype conversion is needed and explain why.

---

## Task 4 — Run `describe(include='all')`

Generate summary statistics for all columns.

In [ ]:
df.describe(include='all').T

**Questions:**

- What is the range of `hh_size` and `age`? Which values look impossible?
- Is `pop_density` plausible across all rows?
- Does `region_code` look standardized (e.g., `01` vs `1` vs missing)?
- Which column appears to mix multiple formats in one field?

---

## Task 5 — Explore categorical columns with `value_counts()`

In [ ]:
df['urban_rural'].value_counts(dropna=False)

In [ ]:
df['province_name'].value_counts(dropna=False)

In [ ]:
# Try another categorical column
df['region_code'].value_counts(dropna=False)

**Questions:**

- Is `urban_rural` consistent, or are there typo categories?
- How many distinct province names appear? Any blanks?
- Are there duplicated categories due to formatting inconsistencies?

---

## Task 6 — Rename columns to `snake_case`

Rename at least 5 columns to clearer names (or keep good names and document why).

In [ ]:
df.columns.tolist()

In [ ]:
df_renamed = df.rename(columns={
    # example edits (add or modify):
    # 'hh_id': 'household_id',
    # 'income_dkw': 'income_dkw_raw',
    # 'survey_date': 'survey_date_raw',
    # 'education_code': 'education_level_code',
    # 'urban_rural': 'settlement_type',
})

df_renamed.head()

**Questions:**

- Would you rename every column in a real pipeline? Why or why not?
- Which 8-10 variables are likely essential for analysis and reporting?

---

## Task 7 — Load only selected columns with `usecols`

With wider datasets, selecting columns at read time saves memory and loading time.

In [ ]:
cols = [
    'hh_id', 'region_code', 'province_name', 'district',
    'urban_rural', 'hh_size', 'income_dkw', 'survey_date',
    'pop_density', 'education_code', 'age'
]

df_small = pd.read_csv(DATA_RAW_DIR / 'datania_households_raw.csv', usecols=cols)
df_small.head()

In [ ]:
print(f'Full DataFrame:  {df.shape}')
print(f'Subset:          {df_small.shape}')
print(f'Memory full:     {df.memory_usage(deep=True).sum() / 1e6:.3f} MB')
print(f'Memory subset:   {df_small.memory_usage(deep=True).sum() / 1e6:.3f} MB')

**Questions:**

- How much memory did you save using `usecols`?
- What happens if a column name in `usecols` is misspelled? Try it and note the error.
- Compare with `df[cols]` after full load: what is the practical difference?

---

## Optional Stretch

1. Standardize `income_dkw` by stripping currency markers/spaces/commas and converting to numeric.
2. Parse `survey_date` with `pd.to_datetime(..., errors='coerce')` and count invalid dates.
3. Flag suspect rows (e.g., `hh_size <= 0`, `age > 120`, duplicated `hh_id`).